# RX-PAD French Medical Prescription Benchmark Analysis

## Executive Summary

**Dataset:** French Medical Prescriptions (Ordonnances Médicales)  
**Total Samples:** 200 prescription images (150 training, 50 testing)  
**Ground Truth Format:** Structured French medical data with fields like medication names, dosages, prescriber info  
**Evaluation Metrics:** Character Error Rate (CER), Word Error Rate (WER), field extraction accuracy

## Benchmark Structure

### Phase P-A: OCR Baseline (Pure Text Extraction)
- **Models:** Azure Document Intelligence, Mistral Document AI
- **Approach:** Direct OCR without language model post-processing
- **Purpose:** Establish baseline OCR performance on French medical prescriptions

### Phase P-B: VLM Baseline (Generic Prompting)
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet  
- **Prompt:** Generic French-aware extraction (no domain context)
- **Purpose:** Evaluate general-purpose VLM capabilities for French medical text

### Phase P-C: VLM + Medical Context
- **Models:** GPT-5-mini, GPT-5-nano, Claude Sonnet
- **Prompt:** Medical prescription-aware prompts with field structure
- **Purpose:** Test if domain-specific context improves VLM performance

## Key Medical Fields (French)

- **exoneration:** Type d'exonération (AFFECTION EXONERANTE, etc.)
- **product_name:** Nom du médicament (JANUMET, ATORVASTATINE, etc.)
- **dosing:** Posologie (50MG/1 000MG, etc.)
- **form:** Forme galénique (CPR, CPR SEC, etc.)
- **timing_when:** Moment de prise (matin, soir, diner)
- **bounds_duration:** Durée du traitement (en mois)
- **prescriber_name:** Nom du prescripteur
- **patient_name:** Nom du patient
- **date_of_prescription:** Date de l'ordonnance

## Analysis Focus Areas

1. **OCR vs VLM:** Do vision language models outperform specialized OCR for French medical text?
2. **Prompting Impact:** How much does medical domain prompting improve VLM performance?
3. **Model Comparison:** GPT-5 vs Claude on French pharmaceutical terminology
4. **Field Extraction:** Can models reliably extract structured prescription information?
5. **French Language Handling:** Accent marks, abbreviations, medical terminology

## To Run This Analysis

1. Ensure consolidation has been run: `python ocr_vs_vlm/results/2_clean/clean_files.py --dataset RX-PAD`
2. This notebook will load pre-consolidated results from `2_clean/RX-PAD/` and generate:
   - CER/WER metrics
   - Comparison visualizations
   - Model performance rankings
   - Medical field extraction analysis
   - French text-specific error analysis

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
from typing import List, Dict, Optional
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

# Import our evaluation metrics
from ocr_vs_vlm.metrics.evaluation_metrics import (
    calculate_cer, calculate_wer
)

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Support French characters in plots
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("✓ Libraries and evaluation metrics loaded successfully!")

## 2. Load Consolidated Results

In [ ]:
# Define paths
RESULTS_DIR = Path("../../../2_clean/RX-PAD")

# Load consolidated results for each phase
phase_dfs = {}

for phase in ['P-A', 'P-B', 'P-C']:
    file_path = RESULTS_DIR / f"{phase}.csv"
    if file_path.exists():
        phase_dfs[phase] = pd.read_csv(file_path)
        print(f"{phase}: {phase_dfs[phase].shape}")
    else:
        print(f"{phase}: Not available")

if not phase_dfs:
    print("\n❌ No consolidated files found. Please run consolidation first:")
    print("   cd ocr_vs_vlm/results/2_clean && python clean_files.py --dataset RX-PAD")

## 3. Data Preview

Quick look at 10 random samples showing ground truth vs model predictions.

In [ ]:
if phase_dfs:
    # Load one phase to show examples
    phase_to_preview = 'P-A'
    df_preview = phase_dfs[phase_to_preview]

    # Get 10 random samples
    random_samples = df_preview.sample(n=min(10, len(df_preview)), random_state=42)

    # Extract columns for preview
    columns_to_show = ['sample_id', 'ground_truth']

    # Add prediction columns (find all columns starting with 'prediction_')
    pred_cols = [col for col in df_preview.columns if col.startswith('prediction_')]
    columns_to_show.extend(pred_cols[:3])  # Show first 3 models

    # Create display dataframe
    display_df = random_samples[columns_to_show].copy()

    # Truncate long strings for readability
    for col in display_df.columns:
        if display_df[col].dtype == 'object':
            display_df[col] = display_df[col].apply(
                lambda x: str(x)[:80] + '...' if pd.notna(x) and len(str(x)) > 80 else x
            )

    print(f"\n{'='*100}")
    print(f"DATA PREVIEW: RX-PAD - {phase_to_preview}")
    print(f"Showing 10 random samples with ground truth and first 3 model predictions")
    print(f"{'='*100}\n")

    display(display_df)

    print(f"\nTotal samples in {phase_to_preview}: {len(df_preview)}")
    print(f"Available models: {', '.join([col.replace('prediction_', '') for col in pred_cols])}")

## 4. Parse and Normalize French Text

Extract prediction columns and normalize French medical text for comparison.

In [ ]:
import re

def normalize_french_text(text: str) -> str:
    """
    Normalize French text for comparison.
    - Removes extra whitespace
    - Normalizes Unicode characters (accents, etc.)
    - Handles common OCR artifacts
    """
    if pd.isna(text) or text is None:
        return ""
    text = str(text).strip()
    # Replace multiple whitespace/newlines with single space
    text = ' '.join(text.split())
    return text.strip()

def get_model_names_from_df(df: pd.DataFrame) -> List[str]:
    """Extract model names from column prefixes."""
    models = set()
    for col in df.columns:
        if col.startswith('prediction_'):
            model = col.replace('prediction_', '')
            models.add(model)
    return sorted(list(models))

# Get model names for each phase
phase_models = {}
for phase, df in phase_dfs.items():
    models = get_model_names_from_df(df)
    phase_models[phase] = models
    phase_label = {
        'P-A': 'OCR baseline',
        'P-B': 'VLM generic',
        'P-C': 'VLM + medical context'
    }.get(phase, phase)
    print(f"{phase} models ({phase_label}): {models}")

## 5. Calculate Character Error Rate (CER) and Word Error Rate (WER)

CER measures the edit distance at the character level, normalized by the length of the ground truth.  
WER measures the edit distance at the word level.

In [ ]:
# Test the functions with French medical text
test_gt = "JANUMET 50MG/1 000MG"
test_pred = "JANUMET 50MG/1000MG"
print(f"Ground truth: '{test_gt}'")
print(f"Prediction: '{test_pred}'")
print(f"CER: {calculate_cer(test_gt, test_pred):.4f}")
print(f"WER: {calculate_wer(test_gt, test_pred):.4f}")

In [ ]:
def calculate_metrics_for_phase(df: pd.DataFrame, models: List[str], phase_name: str) -> pd.DataFrame:
    """Calculate CER and WER for all models in a phase."""
    results = []
    
    for model in models:
        pred_col = f"prediction_{model}"
        time_col = f"inference_time_ms_{model}"
        
        if pred_col not in df.columns:
            continue
        
        for idx, row in df.iterrows():
            gt = row.get('ground_truth', '')
            pred = row.get(pred_col, '')
            inference_time = row.get(time_col, None)
            
            # Skip if no ground truth
            if pd.isna(gt) or normalize_french_text(gt) == '':
                continue
            
            cer = calculate_cer(gt, pred)
            wer = calculate_wer(gt, pred)
            
            results.append({
                'sample_id': row['sample_id'],
                'model': model,
                'phase': phase_name,
                'ground_truth_len': len(normalize_french_text(gt)),
                'prediction_len': len(normalize_french_text(pred)) if not pd.isna(pred) else 0,
                'cer': cer,
                'wer': wer,
                'inference_time_ms': inference_time if not pd.isna(inference_time) else None,
                'has_prediction': not pd.isna(pred) and normalize_french_text(pred) != ''
            })
    
    return pd.DataFrame(results)

# Calculate metrics for each phase
all_metrics = []

phase_labels = {
    'P-A': 'Phase P-A (OCR)',
    'P-B': 'Phase P-B (VLM generic)',
    'P-C': 'Phase P-C (VLM + medical context)'
}

for phase, df in phase_dfs.items():
    models = phase_models.get(phase, [])
    if models:
        print(f"Calculating metrics for {phase}...")
        phase_metrics = calculate_metrics_for_phase(df, models, phase_labels.get(phase, phase))
        print(f"  Computed {len(phase_metrics)} measurements")
        all_metrics.append(phase_metrics)

# Combine all metrics
if all_metrics:
    all_metrics_df = pd.concat(all_metrics, ignore_index=True)
    print(f"\nTotal measurements: {len(all_metrics_df)}")
else:
    all_metrics_df = pd.DataFrame()
    print("No metrics calculated")

## 6. Summary Statistics

Generate summary statistics for CER and WER across all models and phases.

In [ ]:
def summarize_metrics(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """Generate summary statistics for each model and phase."""
    if len(metrics_df) == 0:
        return pd.DataFrame()
    
    agg_dict = {
        'cer': ['mean', 'median', 'std', 'min', 'max'],
        'wer': ['mean', 'median', 'std', 'min', 'max'],
        'has_prediction': ['sum', 'count'],
        'ground_truth_len': 'mean',
        'prediction_len': 'mean',
        'inference_time_ms': ['mean', 'median', 'min', 'max']
    }
    
    summary = metrics_df.groupby(['model', 'phase']).agg(agg_dict).round(4)
    
    # Flatten column names
    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary['prediction_rate'] = (summary['has_prediction_sum'] / summary['has_prediction_count'] * 100).round(1)
    
    return summary.reset_index()

# Generate summary
if len(all_metrics_df) > 0:
    metrics_summary = summarize_metrics(all_metrics_df)
    print("\nModel Performance Summary:")
    print("="*100)
    display(metrics_summary)

## 7. Model Performance Comparison

Compare OCR vs VLM performance on French medical prescriptions.

In [ ]:
if len(all_metrics_df) > 0:
    # Create a clean comparison table
    comparison_cols = ['model', 'phase', 'cer_mean', 'cer_median', 'wer_mean', 'wer_median', 
                       'inference_time_ms_mean', 'prediction_rate']
    comparison_df = metrics_summary[comparison_cols].copy()
    comparison_df.columns = ['Model', 'Phase', 'CER (Mean)', 'CER (Median)', 'WER (Mean)', 'WER (Median)', 
                              'Avg Time (ms)', 'Pred Rate %']
    
    # Sort by CER mean
    comparison_df = comparison_df.sort_values('CER (Mean)')
    print("\nModel Performance Comparison (sorted by CER):")
    print("="*100)
    display(comparison_df)

In [ ]:
# Compare VLM performance across phases (Does medical prompting help?)
if len(all_metrics_df) > 0:
    vlm_phases = ['Phase P-B (VLM generic)', 'Phase P-C (VLM + medical context)']
    vlm_metrics = all_metrics_df[all_metrics_df['phase'].isin(vlm_phases)]
    
    # Models in multiple phases
    model_phase_counts = vlm_metrics.groupby('model')['phase'].nunique()
    multi_phase_models = model_phase_counts[model_phase_counts > 1].index.tolist()
    
    print("\nImpact of Medical Context Prompting on VLM Performance:")
    print("="*80)
    
    for model in sorted(multi_phase_models):
        print(f"\n{model}:")
        model_data = metrics_summary[metrics_summary['model'] == model]
        
        phases_present = model_data['phase'].tolist()
        for i, phase in enumerate(phases_present):
            row = model_data[model_data['phase'] == phase].iloc[0]
            print(f"  {phase}: CER={row['cer_mean']:.4f}, WER={row['wer_mean']:.4f}")
        
        # Calculate improvement from P-B to P-C if available
        if 'Phase P-B (VLM generic)' in phases_present and 'Phase P-C (VLM + medical context)' in phases_present:
            pb_cer = model_data[model_data['phase'] == 'Phase P-B (VLM generic)']['cer_mean'].values[0]
            pc_cer = model_data[model_data['phase'] == 'Phase P-C (VLM + medical context)']['cer_mean'].values[0]
            improvement = ((pb_cer - pc_cer) / pb_cer * 100) if pb_cer > 0 else 0
            if improvement > 0:
                print(f"  ✓ Medical prompting IMPROVED CER by {improvement:.1f}% (P-B → P-C)")
            elif improvement < 0:
                print(f"  ✗ Medical prompting WORSENED CER by {abs(improvement):.1f}% (P-B → P-C)")
            else:
                print(f"  = Medical prompting had NO EFFECT on CER")

## 8. Visualizations

Generate visualizations to compare model performance.

In [ ]:
if len(all_metrics_df) > 0 and len(metrics_summary) > 0:
    # Bar chart: CER by Model and Phase
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Color scheme for phases
    phase_colors = {
        'Phase P-A (OCR)': '#2ecc71',
        'Phase P-B (VLM generic)': '#3498db',
        'Phase P-C (VLM + medical context)': '#e74c3c'
    }
    
    # Plot 1: CER Mean by Model
    ax1 = axes[0]
    x_positions = range(len(metrics_summary))
    bars = ax1.bar(x_positions, metrics_summary['cer_mean'], 
                   color=[phase_colors.get(p, '#95a5a6') for p in metrics_summary['phase']])
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels([f"{row['model']}\n({row['phase'].split()[-1]})" 
                          for _, row in metrics_summary.iterrows()], rotation=45, ha='right')
    ax1.set_ylabel('Character Error Rate (CER)')
    ax1.set_title('CER by Model (Lower is Better)')
    ax1.set_ylim(0, min(1.5, metrics_summary['cer_mean'].max() * 1.2))
    
    # Add value labels on bars
    for bar, val in zip(bars, metrics_summary['cer_mean']):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # Plot 2: WER Mean by Model
    ax2 = axes[1]
    bars = ax2.bar(x_positions, metrics_summary['wer_mean'],
                   color=[phase_colors.get(p, '#95a5a6') for p in metrics_summary['phase']])
    ax2.set_xticks(x_positions)
    ax2.set_xticklabels([f"{row['model']}\n({row['phase'].split()[-1]})" 
                          for _, row in metrics_summary.iterrows()], rotation=45, ha='right')
    ax2.set_ylabel('Word Error Rate (WER)')
    ax2.set_title('WER by Model (Lower is Better)')
    ax2.set_ylim(0, min(1.5, metrics_summary['wer_mean'].max() * 1.2))
    
    # Add value labels on bars
    for bar, val in zip(bars, metrics_summary['wer_mean']):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

In [ ]:
if len(all_metrics_df) > 0:
    # Box plot: CER distribution by model
    # Filter out extreme outliers for better visualization (CER > 2 is almost certainly noise)
    filtered_metrics = all_metrics_df[all_metrics_df['cer'] <= 2.0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Get models ordered by mean CER
    models_ordered = metrics_summary.sort_values('cer_mean')['model'].unique().tolist()
    
    # Box plot for CER
    ax1 = axes[0]
    box_data_cer = [filtered_metrics[filtered_metrics['model'] == m]['cer'].values for m in models_ordered]
    bp1 = ax1.boxplot(box_data_cer, labels=models_ordered, patch_artist=True)
    
    # Color boxes by phase
    phase_colors = {
        'Phase P-A (OCR)': '#2ecc71',
        'Phase P-B (VLM generic)': '#3498db',
        'Phase P-C (VLM + medical context)': '#e74c3c'
    }
    
    for i, model in enumerate(models_ordered):
        phase = metrics_summary[metrics_summary['model'] == model]['phase'].values[0]
        bp1['boxes'][i].set_facecolor(phase_colors.get(phase, '#95a5a6'))
    
    ax1.set_ylabel('Character Error Rate (CER)')
    ax1.set_title('CER Distribution by Model')
    ax1.tick_params(axis='x', rotation=45)
    
    # Box plot for WER
    ax2 = axes[1]
    box_data_wer = [filtered_metrics[filtered_metrics['model'] == m]['wer'].values for m in models_ordered]
    bp2 = ax2.boxplot(box_data_wer, labels=models_ordered, patch_artist=True)
    
    for i, model in enumerate(models_ordered):
        phase = metrics_summary[metrics_summary['model'] == model]['phase'].values[0]
        bp2['boxes'][i].set_facecolor(phase_colors.get(phase, '#95a5a6'))
    
    ax2.set_ylabel('Word Error Rate (WER)')
    ax2.set_title('WER Distribution by Model')
    ax2.tick_params(axis='x', rotation=45)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=p) for p, c in phase_colors.items() 
                       if p in metrics_summary['phase'].values]
    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(0.99, 0.99))
    
    plt.tight_layout()
    plt.show()

In [ ]:
if len(all_metrics_df) > 0:
    # Inference time comparison
    time_summary = metrics_summary[['model', 'phase', 'inference_time_ms_mean']].copy()
    time_summary = time_summary.dropna(subset=['inference_time_ms_mean'])
    
    if len(time_summary) > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        
        x = range(len(time_summary))
        bars = ax.bar(x, time_summary['inference_time_ms_mean'])
        
        ax.set_xticks(x)
        ax.set_xticklabels([f"{row['model']}\n({row['phase'].split()[-1]})" 
                           for _, row in time_summary.iterrows()], rotation=45, ha='right')
        ax.set_ylabel('Average Inference Time (ms)')
        ax.set_title('Inference Time by Model and Phase')
        
        # Add value labels
        for bar, val in zip(bars, time_summary['inference_time_ms_mean']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
                    f'{val:.0f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()

## 9. Medical Field Extraction Analysis

Analyze how well models extract specific French medical prescription fields.

In [ ]:
# Common fields in French medical prescriptions
MEDICAL_FIELDS_FR = {
    'exoneration': 'Exemption Type',
    'product_name': 'Medication Name',
    'dosing': 'Dosage',
    'form': 'Form (tablet/liquid)',
    'container_quantity_inside': 'Container Quantity',
    'dose_quantity_low': 'Dose Amount',
    'dose_unit': 'Dose Unit',
    'timing_when': 'Timing (morning/evening)',
    'bounds_duration': 'Duration',
    'structure_name': 'Healthcare Facility',
    'prescriber_name': 'Prescriber Name',
    'patient_name': 'Patient Name',
    'date_of_prescription': 'Prescription Date'
}

def check_field_presence(text: str, field: str) -> bool:
    """Check if a field is present in the text."""
    if pd.isna(text):
        return False
    return field in str(text)

# Only run if we have data
if len(phase_dfs) > 0:
    # Analyze field extraction for each phase
    for phase in ['P-C', 'P-B', 'P-A']:
        if phase not in phase_dfs:
            continue
            
        df = phase_dfs[phase]
        models = phase_models.get(phase, [])
        
        if not models:
            continue
            
        print(f"\nField Extraction Analysis ({phase}):")
        print("="*80)
        
        # Analyze ground truth first
        print("\nFields present in Ground Truth:")
        for field_key, field_en in MEDICAL_FIELDS_FR.items():
            count = df['ground_truth'].apply(lambda x: check_field_presence(x, field_key)).sum()
            pct = count / len(df) * 100
            if count > 0:
                print(f"  {field_key} ({field_en}): {count}/{len(df)} ({pct:.1f}%)")
        
        # Analyze each model
        for model in models:
            pred_col = f"prediction_{model}"
            if pred_col not in df.columns:
                continue
            
            print(f"\nFields extracted by {model}:")
            for field_key, field_en in MEDICAL_FIELDS_FR.items():
                gt_count = df['ground_truth'].apply(lambda x: check_field_presence(x, field_key)).sum()
                pred_count = df[pred_col].apply(lambda x: check_field_presence(x, field_key)).sum()
                if gt_count > 0:
                    recall = pred_count / gt_count * 100
                    print(f"  {field_key}: {pred_count}/{gt_count} ({recall:.1f}% recall)")
        
        break  # Only analyze one phase

## 10. Sample-Level Analysis

Examine specific samples to understand where models succeed or fail.

In [ ]:
if len(all_metrics_df) > 0:
    # Find best and worst performing samples
    # Pivot to compare same samples across models
    sample_pivot = all_metrics_df.pivot_table(
        index='sample_id', 
        columns='model', 
        values='cer',
        aggfunc='first'
    ).reset_index()
    
    # Calculate mean CER across all models for each sample
    model_cols = [c for c in sample_pivot.columns if c != 'sample_id']
    sample_pivot['mean_cer'] = sample_pivot[model_cols].mean(axis=1)
    sample_pivot['std_cer'] = sample_pivot[model_cols].std(axis=1)
    
    # Best samples (lowest mean CER)
    print("\nTop 10 EASIEST Samples (lowest mean CER across all models):")
    print("="*80)
    print(sample_pivot.nsmallest(10, 'mean_cer')[['sample_id', 'mean_cer', 'std_cer']].to_string(index=False))
    
    print("\n")
    print("Top 10 HARDEST Samples (highest mean CER across all models):")
    print("="*80)
    print(sample_pivot.nlargest(10, 'mean_cer')[['sample_id', 'mean_cer', 'std_cer']].to_string(index=False))

## 11. Export Results

Save the analysis results for further use.

In [ ]:
# Create output directory
OUTPUT_DIR = Path("../../../4_postprocessing/RX-PAD")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if len(all_metrics_df) > 0:
    # Save detailed metrics
    all_metrics_df.to_csv(OUTPUT_DIR / 'detailed_metrics.csv', index=False)
    print(f"Saved detailed metrics to: {OUTPUT_DIR / 'detailed_metrics.csv'}")
    
    # Save summary comparison
    if 'comparison_df' in dir():
        comparison_df.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
        print(f"Saved model comparison to: {OUTPUT_DIR / 'model_comparison.csv'}")
    
    # Save sample-level pivot
    if 'sample_pivot' in dir():
        sample_pivot.to_csv(OUTPUT_DIR / 'sample_level_cer.csv', index=False)
        print(f"Saved sample-level CER to: {OUTPUT_DIR / 'sample_level_cer.csv'}")
    
    print("\nAll exports complete!")
else:
    print("No data to export. Run consolidation first.")

## 12. Summary & Conclusions

### Key Findings:

1. **OCR vs VLM Performance**: Compare the CER/WER metrics between Phase P-A (OCR) and Phase P-B/P-C (VLM) models for French medical prescriptions.

2. **Impact of Medical Context**: Compare Phase P-B (generic prompts) vs Phase P-C (medical prescription prompts) to determine if domain-specific prompting improves VLM performance on pharmaceutical terminology.

3. **Best Performing Models**: Identify which models achieve the lowest error rates on French medical prescription text.

4. **Medical Field Extraction**: Evaluate how well models capture structured information like medication names, dosages, prescriber details, and patient information.

### Special Considerations for French Medical Prescriptions:
- French medical terminology and abbreviations (CPR, CPR SEC, etc.)
- Medication names (brand names vs generic)
- Dosage units and quantities
- Handwritten vs printed prescriptions
- Accent marks and special characters in French

### Next Steps:
- Run additional VLM models to expand comparison
- Test different prompting strategies for French pharmaceutical terminology
- Analyze failure cases in detail
- Compare with specialized French medical OCR engines